
# 🔎 Mini RAG Agent (Free) — Local Slides + Web Search

This notebook lets you test a **free prototype** of a retrieval-augmented agent that can:
- Search your **local lecture slides** (PDFs) ✅
- Search the **web via DuckDuckGo** (free) ✅
- Synthesize an answer using a **small local model** (FLAN-T5-small) ✅

> **No paid keys required.** You can run it on CPU. Internet is needed for web search.


## 1) Install dependencies

In [3]:

!pip -q install duckduckgo-search==6.1.10 sentence-transformers==3.0.1 faiss-cpu==1.8.0.post1 \
transformers==4.44.2 accelerate==0.34.2 pdfplumber==0.11.4 trafilatura==1.7.0 beautifulsoup4==4.12.3 tqdm==4.66.4


ERROR: Could not find a version that satisfies the requirement duckduckgo-search==6.1.10 (from versions: 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.9.5, 1.0, 1.1, 1.2, 1.3, 1.3.5, 1.4, 1.5, 1.5.1, 1.5.2, 1.6, 1.6.2, 1.7.1, 1.8, 1.8.1, 1.8.2, 2.0.2, 2.1.3, 2.2.0, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.5.0, 2.6.0, 2.6.1, 2.7.0, 2.8.0, 2.8.1, 2.8.3, 2.8.4, 2.8.5, 2.8.6, 2.9.0, 2.9.1, 2.9.2, 2.9.3, 2.9.4, 2.9.5, 3.0.2, 3.1.1, 3.2.0, 3.3.0, 3.4.1, 3.5.0, 3.6.0, 3.7.0, 3.7.1, 3.8.0, 3.8.1, 3.8.2, 3.8.3, 3.8.4, 3.8.5, 3.8.6, 3.9.3, 3.9.4, 3.9.5, 3.9.6, 3.9.8, 3.9.9, 3.9.10, 3.9.11, 4.0.0, 4.1.0, 4.1.1, 4.2, 4.3, 4.3.1, 4.4, 4.4.1, 4.4.2, 4.4.3, 4.5.0, 5.0b0, 5.0b1, 5.0, 5.1.0, 5.2.0, 5.2.1, 5.2.2, 5.3.0b1, 5.3.0b2, 5.3.0b3, 5.3.0b4, 5.3.0, 5.3.1b1, 5.3.1, 6.0.0, 6.1.0, 6.1.1, 6.1.2, 6.1.3, 6.1.4, 6.1.5, 6.1.6, 6.1.7, 6.1.8, 6.1.9, 6.1.11, 6.1.12, 6.2.0, 6.2.1, 6.2.2, 6.2.3, 6.2.4, 6.2.5, 6.2.6, 6.2.7, 6.2.8b1, 6.2.8, 6.2.9, 6.2.10, 6.2.11b1, 6.2.12, 6.2.13, 6.3.0, 6.3.1, 6.3.2, 6.3.3, 6.3.4, 6.3.5, 6.3.6, 6

## 2) Configuration

In [ ]:

from pathlib import Path

DATA_DIR = Path('data/slides')  # put your PDFs here

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Retrieval knobs

CHUNK_SIZE = 800   # characters per chunk

CHUNK_OVERLAP = 120

TOP_K_LOCAL = 4

TOP_K_WEB = 3

MAX_WEB_CHARS = 4000

DEVICE = 'cuda'  # set to 'cpu' if you don't have a GPU

print(f'Put your PDFs in: {DATA_DIR.resolve()}')



## 3) Ingest & index local PDFs (slides)

In [ ]:

import pdfplumber, re, json, os

from tqdm import tqdm


def clean_text(t:str)->str:

    t = re.sub(r'\s+', ' ', t)

    return t.strip()


def load_pdfs_text(folder:Path):

    docs = []

    for pdf in sorted(folder.glob('*.pdf')):

        try:

            with pdfplumber.open(pdf) as p:

                pages = [clean_text(page.extract_text() or '') for page in p.pages]

            text = '\n'.join([x for x in pages if x])

            if text:

                docs.append({'path': str(pdf), 'text': text})

                print(f'Loaded: {pdf.name} ({len(text)} chars)')

        except Exception as e:

            print(f'Failed {pdf.name}: {e}')

    return docs


def chunk_text(text, chunk_size=800, overlap=120):

    chunks = []

    start = 0

    while start < len(text):

        end = min(len(text), start + chunk_size)

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


docs = load_pdfs_text(DATA_DIR)

all_chunks, meta = [], []

for d in docs:

    chunks = chunk_text(d['text'], CHUNK_SIZE, CHUNK_OVERLAP)

    for i, c in enumerate(chunks):

        all_chunks.append(c)

        meta.append({'source': d['path'], 'chunk_id': i})

print(f'Total chunks: {len(all_chunks)}')



## 4) Build vector index (Sentence-Transformers + FAISS)

In [ ]:

from sentence_transformers import SentenceTransformer

import faiss, numpy as np


if not all_chunks:

    print('⚠️ No PDFs found yet. Add some files to data/slides and rerun the previous cell.')



emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)

embs = emb_model.encode(all_chunks, normalize_embeddings=True, batch_size=64, convert_to_numpy=True)

index = faiss.IndexFlatIP(embs.shape[1])  # cosine via normalized vectors

index.add(embs)

print('FAISS index built:', index.ntotal, 'chunks')



## 5) Web search (DuckDuckGo) + content extraction

In [ ]:

from duckduckgo_search import DDGS

import trafilatura


def web_search_and_extract(query, k=3, max_chars=4000):

    results = []

    with DDGS() as ddgs:

        for r in ddgs.text(query, region='us-en', safesearch='moderate', max_results=k):

            url = r.get('href') or r.get('url')

            title = r.get('title')

            if not url:

                continue

            try:

                downloaded = trafilatura.fetch_url(url)

                text = trafilatura.extract(downloaded) or ''

                text = text[:max_chars]

                if text:

                    results.append({'title': title, 'url': url, 'text': text})

            except Exception as e:

                pass

    return results


# quick smoke test (will only work with internet access)

# web_hits = web_search_and_extract('What is Business Process Reengineering in EHR?', k=2)

# web_hits[:1]



## 6) Local retrieval helper

In [ ]:

def retrieve_local(query, top_k=4):

    q_emb = emb_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)

    D, I = index.search(q_emb, top_k)

    items = []

    for score, idx in zip(D[0], I[0]):

        items.append({

            'score': float(score),

            'text': all_chunks[idx],

            'meta': meta[idx]

        })

    return items



## 7) Answer synthesis (FLAN‑T5‑small, local)

In [ ]:

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import torch


gen_model_name = 'google/flan-t5-small'

tokenizer = AutoTokenizer.from_pretrained(gen_model_name)

gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

gen_model = gen_model.to(DEVICE)



def generate_answer(question, local_contexts, web_contexts, max_new_tokens=220):

    # build compact prompt

    ctx_blocks = []

    for c in local_contexts:

        ctx_blocks.append(f"[SLIDE {Path(c['meta']['source']).name}#{c['meta']['chunk_id']}] {c['text']}")

    for i, w in enumerate(web_contexts):

        ctx_blocks.append(f"[WEB {i+1} {w['title']}] {w['text']}")

    context = '\n\n'.join(ctx_blocks)[:6000]

    prompt = (

        "You are a helpful teaching assistant. Use the CONTEXT to answer the QUESTION in 5-7 sentences. "

        "Cite sources inline like [SLIDE filename#id] or [WEB n]. If unsure, say what is missing.\n\n"

        f"CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER:"

    )

    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():

        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.4, do_sample=False)

    return tokenizer.decode(out[0], skip_special_tokens=True)



## 8) Ask a question (end‑to‑end)

In [ ]:

question = "Why is Business Process Re-engineering important when implementing an EHR?"  # change me

# Retrieve local
local_hits = retrieve_local(question, top_k=TOP_K_LOCAL)
# Retrieve web
try:
    web_hits = web_search_and_extract(question, k=TOP_K_WEB, max_chars=MAX_WEB_CHARS)
except Exception:
    web_hits = []

print(f"Local hits: {len(local_hits)} | Web hits: {len(web_hits)}")
answer = generate_answer(question, local_hits, web_hits)
print('\n' + answer)


## 9) (Optional) Simple UI with Gradio

In [ ]:

try:

    import gradio as gr

    def rag_qa(q):

        local_hits = retrieve_local(q, top_k=TOP_K_LOCAL)

        try:

            web_hits = web_search_and_extract(q, k=TOP_K_WEB, max_chars=MAX_WEB_CHARS)

        except Exception:

            web_hits = []

        ans = generate_answer(q, local_hits, web_hits)

        sources = []

        for h in local_hits:

            sources.append(f"SLIDE: {Path(h['meta']['source']).name}#{h['meta']['chunk_id']} (score={h['score']:.2f})")

        for i, w in enumerate(web_hits):

            sources.append(f"WEB {i+1}: {w['title']} — {w['url']}")

        return ans, "\n".join(sources)


    demo = gr.Interface(fn=rag_qa, inputs=gr.Textbox(label="Question"), outputs=[gr.Textbox(label="Answer"), gr.Textbox(label="Sources")], title="Mini RAG Agent (Slides + Web)")

    # Uncomment to launch UI

    # demo.launch()

except Exception as e:

    print('Gradio not available (optional).')

